# DPO Evaluation — Did the Trained Model Actually Improve?

**Purpose:** measure whether the LoRA-DPO-trained model from the previous notebook meaningfully reduces over-formatting (on the test split) while preserving structure where structure helps (on the adversarial set).

## What we measure

**Structural metrics on generated responses.** For each prompt, we compute:
- Bullet markers (lines starting with `-`, `*`, or `1.`)
- Bold markers (`**`)
- Header markers (lines starting with `#`)
- Emoji count
- Word count

We compute these for both the **base model** and the **trained model**, then compare each to the **gold `chosen` response** from the dataset. If the trained model's structural profile is closer to the gold than the base model's is, training improved things.

## Two ways to load the trained model

**Option A (recommended): from HuggingFace.** If you've pushed the adapter to HF (uncommented cells 12 in `dpo_02_train.ipynb`), this notebook can load it from anywhere.

**Option B: same Kaggle session as training.** If you haven't pushed, run this notebook **in the same Kaggle session** as the training notebook — `trainer.model` will still be in memory. Skip cells 4-5 in that case and use the existing `trained_model` variable.

**Expected runtime:** ~20-25 minutes on a Kaggle T4.

---
*Part of the [Prosify project](https://github.com/[your-username]/prosify).*

## 1. Install dependencies

In [ ]:
!pip install -q -U transformers trl datasets accelerate peft bitsandbytes torchao

import transformers, trl, datasets, accelerate, torch, peft
print(f"transformers: {transformers.__version__}")
print(f"trl:          {trl.__version__}")
print(f"peft:         {peft.__version__}")
print(f"torch:        {torch.__version__}")

## 2. Configuration

Set your HF username and the adapter repo (where you pushed the trained adapter).

In [ ]:
# ===== EDIT THESE =====
HF_USERNAME = "your-hf-username"
ADAPTER_REPO = f"{HF_USERNAME}/prosify-qwen-1.5b-lora"  # change if you used a different name
# ======================

DATASET_REPO = f"{HF_USERNAME}/formatbench"
BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
SEED = 42
MAX_NEW_TOKENS = 200

# If you DIDN'T push the adapter and are running this in the same Kaggle session
# as the training notebook, set this to True. It will use the in-memory model.
USE_IN_MEMORY_MODEL = False

## 3. Confirm GPU

In [ ]:
import torch, random, numpy as np

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:            {torch.cuda.get_device_name(0)}")
else:
    raise RuntimeError("No GPU detected.")

## 4. Load the base model and trained adapter

Skip this cell if `USE_IN_MEMORY_MODEL = True`. Otherwise this loads the base model and applies your trained LoRA adapter.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

if not USE_IN_MEMORY_MODEL:
    print(f"Loading base model {BASE_MODEL}...")
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        dtype=torch.bfloat16,
        device_map="auto",
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        base_model.config.pad_token_id = tokenizer.eos_token_id

    print(f"\nLoading trained adapter from {ADAPTER_REPO}...")
    trained_model = PeftModel.from_pretrained(base_model, ADAPTER_REPO)
    trained_model.eval()

    # Load a SEPARATE copy of the base model for comparison (without adapter)
    print(f"\nLoading a separate base model copy for comparison...")
    base_only_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        dtype=torch.bfloat16,
        device_map="auto",
    )
    base_only_model.eval()

    print(f"\n✓ Both models loaded.")
    print(f"  GPU mem in use: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
else:
    print("USE_IN_MEMORY_MODEL=True — assuming `trained_model`, `base_only_model`, `tokenizer` exist from training notebook")

## 5. Load the test split and adversarial set

Recreate the test split with the same seed used in earlier notebooks, then load the adversarial set from its config.

In [ ]:
from datasets import load_dataset
from collections import defaultdict

# Main test split (recreate with same seed)
raw = load_dataset(DATASET_REPO, split="train")
rng = random.Random(SEED)
by_context = defaultdict(list)
for row in raw:
    by_context[row["context"]].append(row)

test_rows = []
for context, items in sorted(by_context.items()):
    shuffled = items[:]
    rng.shuffle(shuffled)
    n = len(shuffled)
    n_test = max(1, n // 10)
    n_val = max(1, n // 10)
    n_train = n - n_test - n_val
    test_rows.extend(shuffled[n_train + n_val:])
rng.shuffle(test_rows)

# Adversarial set
adversarial_rows = list(load_dataset(DATASET_REPO, "adversarial", split="test"))

print(f"Main test split: {len(test_rows)} examples")
print(f"Adversarial set: {len(adversarial_rows)} examples")

## 6. Generation helper

Same generation logic as the training notebook — chat-template formatted, deterministic (no sampling).

In [ ]:
def generate_response(model, prompt, max_new_tokens=MAX_NEW_TOKENS):
    messages = [{"role": "user", "content": prompt}]
    formatted = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    return tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

# Quick sanity check
test_prompt = test_rows[0]["prompt"]
print(f"Sanity check — generating one response from each model...")
base_test = generate_response(base_only_model, test_prompt, max_new_tokens=50)
trained_test = generate_response(trained_model, test_prompt, max_new_tokens=50)
print(f"\nPrompt: {test_prompt[:80]}...")
print(f"Base:    {base_test[:80]}...")
print(f"Trained: {trained_test[:80]}...")

## 7. Generate responses on the main test set

This is the slow part. ~10-12 minutes for ~50 examples × 2 models.

In [ ]:
from tqdm.auto import tqdm

test_results = []
for row in tqdm(test_rows, desc="Generating on test split"):
    base_resp = generate_response(base_only_model, row["prompt"])
    trained_resp = generate_response(trained_model, row["prompt"])
    test_results.append({
        "prompt": row["prompt"],
        "context": row["context"],
        "gold_chosen": row["chosen"],
        "gold_rejected": row["rejected"],
        "base_response": base_resp,
        "trained_response": trained_resp,
    })

print(f"\n✓ Generated {len(test_results)} test responses")

## 8. Generate responses on the adversarial set

Another ~15-18 minutes.

In [ ]:
adversarial_results = []
for row in tqdm(adversarial_rows, desc="Generating on adversarial set"):
    base_resp = generate_response(base_only_model, row["prompt"])
    trained_resp = generate_response(trained_model, row["prompt"])
    adversarial_results.append({
        "prompt": row["prompt"],
        "topic_area": row.get("topic_area", "unknown"),
        "gold_chosen": row["chosen"],
        "gold_rejected": row["rejected"],
        "base_response": base_resp,
        "trained_response": trained_resp,
    })

print(f"\n✓ Generated {len(adversarial_results)} adversarial responses")

## 9. Structural metrics

Quantify the formatting characteristics of each response: bullets, bold markers, headers, emojis, word count. We'll compare base vs trained against the gold chosen responses.

In [ ]:
import re

# Emoji regex (matches most common emoji unicode ranges)
EMOJI_RE = re.compile(
    "["
    "\U0001F600-\U0001F64F"  # emoticons
    "\U0001F300-\U0001F5FF"  # symbols & pictographs
    "\U0001F680-\U0001F6FF"  # transport & map
    "\U0001F1E0-\U0001F1FF"  # flags
    "\U00002500-\U00002BEF"  # chinese char
    "\U00002702-\U000027B0"
    "\U0001f926-\U0001f937"
    "\U00010000-\U0010ffff"
    "\u2640-\u2642"
    "\u2600-\u2B55"
    "\u200d"
    "\u23cf"
    "\u23e9"
    "\u231a"
    "\ufe0f"
    "\u3030"
    "]+",
    flags=re.UNICODE,
)

def compute_metrics(text: str) -> dict:
    """Return structural metrics for a single response."""
    if not text:
        return {"bullets": 0, "bold": 0, "headers": 0, "emojis": 0, "words": 0}

    # Bullets: lines starting with -, *, or a number followed by .
    bullet_lines = re.findall(r"^[\s]*([-*]|\d+\.)\s", text, flags=re.MULTILINE)

    # Bold: pairs of **
    bold_count = text.count("**") // 2

    # Headers: lines starting with # (markdown) OR lines that look like **Header:**
    md_headers = len(re.findall(r"^#+\s", text, flags=re.MULTILINE))
    bold_label_headers = len(re.findall(r"\*\*[A-Z][^*]+:\*\*", text))
    header_count = md_headers + bold_label_headers

    # Emojis
    emoji_count = len(EMOJI_RE.findall(text))

    # Words
    word_count = len(text.split())

    return {
        "bullets": len(bullet_lines),
        "bold": bold_count,
        "headers": header_count,
        "emojis": emoji_count,
        "words": word_count,
    }

# Sanity check
sample_metrics_base = compute_metrics(test_results[0]["base_response"])
sample_metrics_trained = compute_metrics(test_results[0]["trained_response"])
sample_metrics_gold = compute_metrics(test_results[0]["gold_chosen"])
print(f"Sample (test[0]):")
print(f"  base:    {sample_metrics_base}")
print(f"  trained: {sample_metrics_trained}")
print(f"  gold:    {sample_metrics_gold}")

## 10. Aggregate metrics — main test split

On the test split, the gold responses are PROSE. So we want trained model metrics to be **lower** than base model metrics on `bullets`, `bold`, `headers`, `emojis` — and ideally close to gold.

In [ ]:
import statistics

def aggregate(results, key):
    """Aggregate one of (base_response, trained_response, gold_chosen) across rows."""
    metrics_list = [compute_metrics(r[key]) for r in results]
    return {
        m: statistics.mean(d[m] for d in metrics_list)
        for m in ["bullets", "bold", "headers", "emojis", "words"]
    }

test_base = aggregate(test_results, "base_response")
test_trained = aggregate(test_results, "trained_response")
test_gold = aggregate(test_results, "gold_chosen")
test_gold_rej = aggregate(test_results, "gold_rejected")

print("=== MAIN TEST SPLIT — MEAN STRUCTURAL METRICS PER RESPONSE ===\n")
print(f"{'metric':<10} {'base':>10} {'trained':>10} {'gold':>10} {'(rejected)':>12}")
print("-" * 60)
for m in ["bullets", "bold", "headers", "emojis", "words"]:
    print(f"{m:<10} {test_base[m]:>10.2f} {test_trained[m]:>10.2f} {test_gold[m]:>10.2f} {test_gold_rej[m]:>12.2f}")

print("\nInterpretation:")
print("  - Trained closer to GOLD than BASE → training helped")
print("  - Trained similar to BASE → training had minimal effect")
print("  - Trained closer to REJECTED → reward hacking went wrong direction")

## 11. Aggregate metrics — adversarial set

On the adversarial set, the gold responses USE STRUCTURE (recipes, install steps, comparisons). So we want trained model metrics to be **similar to or higher than** base on `bullets`, `headers`, etc. — close to gold. A model that strips structure here has reward-hacked.

In [ ]:
adv_base = aggregate(adversarial_results, "base_response")
adv_trained = aggregate(adversarial_results, "trained_response")
adv_gold = aggregate(adversarial_results, "gold_chosen")
adv_gold_rej = aggregate(adversarial_results, "gold_rejected")

print("=== ADVERSARIAL SET — MEAN STRUCTURAL METRICS PER RESPONSE ===\n")
print(f"{'metric':<10} {'base':>10} {'trained':>10} {'gold':>10} {'(rejected)':>12}")
print("-" * 60)
for m in ["bullets", "bold", "headers", "emojis", "words"]:
    print(f"{m:<10} {adv_base[m]:>10.2f} {adv_trained[m]:>10.2f} {adv_gold[m]:>10.2f} {adv_gold_rej[m]:>12.2f}")

print("\nInterpretation:")
print("  - Trained similar to GOLD (and BASE) → context-sensitivity preserved ✓")
print("  - Trained dropped structure markers significantly → reward hacking happened")
print("  - Trained added MORE structure than base → unexpected but not bad")

## 12. Win rate via structural distance to gold

For each prompt, we measure how structurally close each model's response is to the gold chosen response. The model whose response is closer wins. This gives us a 'win rate' for the trained model vs the base.

In [ ]:
def structural_distance(metrics_a: dict, metrics_b: dict) -> float:
    """Sum of absolute differences across the formatting markers.
    Words is excluded since the absolute difference would dominate."""
    keys = ["bullets", "bold", "headers", "emojis"]
    return sum(abs(metrics_a[k] - metrics_b[k]) for k in keys)

def compute_win_rate(results, label=""):
    trained_wins = 0
    base_wins = 0
    ties = 0
    for r in results:
        gold_m = compute_metrics(r["gold_chosen"])
        base_m = compute_metrics(r["base_response"])
        trained_m = compute_metrics(r["trained_response"])
        d_base = structural_distance(base_m, gold_m)
        d_trained = structural_distance(trained_m, gold_m)
        if d_trained < d_base:
            trained_wins += 1
        elif d_trained > d_base:
            base_wins += 1
        else:
            ties += 1
    total = len(results)
    print(f"=== {label} WIN RATE (lower structural distance to gold) ===")
    print(f"  Trained wins: {trained_wins}/{total}  ({trained_wins/total:.1%})")
    print(f"  Base wins:    {base_wins}/{total}  ({base_wins/total:.1%})")
    print(f"  Ties:         {ties}/{total}  ({ties/total:.1%})")
    return trained_wins, base_wins, ties

print("Main test set:\n")
test_w, test_b, test_t = compute_win_rate(test_results, "MAIN TEST")
print("\nAdversarial set:\n")
adv_w, adv_b, adv_t = compute_win_rate(adversarial_results, "ADVERSARIAL")

## 13. Qualitative samples — main test split

5 random side-by-side examples. The numbers above are aggregate; these let you eyeball whether the trained model's outputs are actually different from the base.

In [ ]:
_rng = random.Random(SEED)
sample_idxs = _rng.sample(range(len(test_results)), min(5, len(test_results)))

for i, idx in enumerate(sample_idxs):
    r = test_results[idx]
    print(f"--- TEST SAMPLE {i+1} (context: {r['context']}) ---\n")
    print(f"PROMPT: {r['prompt']}\n")
    print(f"BASE RESPONSE:\n{r['base_response'][:500]}\n")
    print(f"TRAINED RESPONSE:\n{r['trained_response'][:500]}\n")
    print(f"GOLD CHOSEN:\n{r['gold_chosen'][:500]}\n")
    print("=" * 60 + "\n")

## 14. Qualitative samples — adversarial set

Same idea, on the adversarial set. The trained model should ideally **keep** structure here.

In [ ]:
_rng = random.Random(SEED + 1)
adv_sample_idxs = _rng.sample(range(len(adversarial_results)), min(5, len(adversarial_results)))

for i, idx in enumerate(adv_sample_idxs):
    r = adversarial_results[idx]
    print(f"--- ADVERSARIAL SAMPLE {i+1} (topic: {r['topic_area']}) ---\n")
    print(f"PROMPT: {r['prompt']}\n")
    print(f"BASE RESPONSE:\n{r['base_response'][:500]}\n")
    print(f"TRAINED RESPONSE:\n{r['trained_response'][:500]}\n")
    print(f"GOLD CHOSEN:\n{r['gold_chosen'][:500]}\n")
    print("=" * 60 + "\n")

## 15. Final summary

Putting it all together.

In [ ]:
print("=" * 60)
print("  PROSIFY v1 EVALUATION SUMMARY")
print("=" * 60)
print()
print(f"Base model:    {BASE_MODEL}")
print(f"Trained model: {BASE_MODEL} + LoRA from {ADAPTER_REPO}")
print()
print("--- MAIN TEST SPLIT (49 examples, gold = prose) ---")
print(f"  Trained win rate: {test_w}/{test_w + test_b + test_t} ({test_w/(test_w + test_b + test_t):.1%})")
print(f"  Mean bullets per response — Base:{test_base['bullets']:.2f}  Trained:{test_trained['bullets']:.2f}  Gold:{test_gold['bullets']:.2f}")
print(f"  Mean headers per response — Base:{test_base['headers']:.2f}  Trained:{test_trained['headers']:.2f}  Gold:{test_gold['headers']:.2f}")
print()
print("--- ADVERSARIAL SET (80 examples, gold = structure) ---")
print(f"  Trained win rate: {adv_w}/{adv_w + adv_b + adv_t} ({adv_w/(adv_w + adv_b + adv_t):.1%})")
print(f"  Mean bullets per response — Base:{adv_base['bullets']:.2f}  Trained:{adv_trained['bullets']:.2f}  Gold:{adv_gold['bullets']:.2f}")
print(f"  Mean headers per response — Base:{adv_base['headers']:.2f}  Trained:{adv_trained['headers']:.2f}  Gold:{adv_gold['headers']:.2f}")
print()
print("=" * 60)
print()
print("Possible outcomes:")
print()
print("  STRONG SUCCESS:")
print("    Main test win rate > 60% AND adversarial win rate > 60%")
print("    → Trained model meaningfully improved on the target task while preserving")
print("      context-sensitive structure. Publish as is.")
print()
print("  MIXED RESULT:")
print("    Main test win rate > 60% but adversarial < 40%")
print("    → Reward hacking confirmed. Need higher beta or fewer epochs and retrain.")
print()
print("  NO MEANINGFUL CHANGE:")
print("    Main test win rate 40-55% and adversarial 40-55%")
print("    → LoRA + DPO at this rank / epoch count is too weak to shift generation.")
print("      Try higher LoRA rank (64 or 128) and 2-3 epochs.")
print()
print("  REGRESSION:")
print("    Main test win rate < 40%")
print("    → Something went wrong in training. Re-check hyperparameters.")